# Project: Medbot

**Overview:** Develop a medical chatbot to assist users.

**Task:** Chatbot / NLP
**Dataset:** Medical texts


**Just to give you a heads up:** We won't be having a model performing like ChatGPT or Bard, but at least we will have an idea about how we can create our own smaller versions of such powerful LLMs.  

## Importing and Installing Libraries/Packages
We will start by installing our necessary packages.

**bitsandbytes**: This package will allow us to run 4bit quantization on our model

**transformers**: This Hugging Face package will allow us to load state-of-the-art models easily into our notebook

**peft**: This package allows us to add PEFT techniques easily to our model, such as LoRA

**accelerate**: Accelerate is a handy package that allows us to run boiler plate code with a few lines of code

**datasets**: This package allows us to easily import datasets from the Hugging Face platform to be directly used

In [ ]:
!pip install bitsandbytes
!pip install git+https://github.com/huggingface/transformers.git
!pip install git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.7 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-3sxynjax
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-3sxynjax
  Resolved https://github.com/huggingface/transformers.git to commit d64a6d67d8c004a25570db4df5689e06caea6af7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11327720 sha256=5f9e9dcef0f768e73d4af9cf571d69d2b9256ae906f4a24f585a03b4058679f6
  Stored in directory: /tmp/pip-ephem-wheel-cache-96j_qkm2/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
  

In [ ]:
import torch
import transformers
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

## Loading our model

Let's start by loading our model. We will use the GPT Neox 20b Model by EleutherAI!

In [ ]:
hf_model = "EleutherAI/gpt-neox-20b"

We will also set the bitsandbytes configurations needed for our model to run on our single colab GPU. The needed paramaters will be 'Double Quantization' 'Quantization Type' and the computational type needs to be set to bfloat16.

In [ ]:
bitsbytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # Double Quantization
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    # Computational Type
    bnb_4bit_compute_dtype=torch.bfloat16
    ) #Test Your Zaka


We will then set our tokenizer, and our model using the AutoTokenizer and AutoModelforCausalLM classes

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = AutoModelForCausalLM.from_pretrained(
    hf_model,
    device_map="auto",
    quantization_config=bitsbytes_config,
)

model.resize_token_embeddings(len(tokenizer))

#Test Your Zaka


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 46 files:   0%|          | 0/46 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/532 [00:00<?, ?it/s]

Embedding(50278, 6144)

## Model Preprocessing

We now have to apply some preprocessing to our model so we can prepare it for training. First we need to further reduce our memory consumption by using the gradient_checkpointing_enable() fucntion on our model. We then use the prepare_model_for_kbit_training function so that we can use 4bit quantization training.

In [ ]:
model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)
#Test Your Zaka

Explain with your own words how 4-bit quantization affects accuracy.

4 bit quantization make the model lighter so it can run on limited hardware so accuracy is decreased but not much as large models have lots of redundancy. Large models store weights in a very precise numbers, when these numbers are rounded using 4-bit quantization their will be a slight accuracy drop.

**Test Your Zaka**

We will also set a function that will print the number of trainable parameters our model has.

In [ ]:
def print_trainable_parameters(model):
    trainable_parameters = 0
    all_paramaters = 0
    for _, param in model.named_parameters():
        all_paramaters += param.numel()
        if param.requires_grad:
            trainable_parameters += param.numel()
    print(
        f"Trainable: {trainable_parameters} || All: {all_paramaters} || Trainable %: {100 * trainable_parameters / all_paramaters}"
    )

Finally we will set the configurations for our LoRA. The paramaters needed are the rank updates, the default LoRa alpha value, the target modules which need to be set to query_key_value, the default lora dropout rate, bias should be set to none, and the task type according to the model we are using.

In [ ]:
config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
    #Test Your Zaka

)

# Insert the configs above to the model using the get_peft_model function
model = get_peft_model(model, config)
#Test Your Zaka


# Print the trainable parameters of the model
print_trainable_parameters(model)

Trainable: 34603008 || All: 10621612032 || Trainable %: 0.3257792498516293


## Dataset Loading

Let's load our medical dataset from Hugging Face. We will use the `medalpaca/medical_meadow_wikidoc_patient_information` dataset. You can access it [here](https://huggingface.co/datasets/medalpaca/medical_meadow_wikidoc).

In [ ]:
#Test Your Zaka
dataset_name = "medalpaca/medical_meadow_wikidoc_patient_information"
dataset = load_dataset(dataset_name)

# Mapping the needed column as our data using a lambda statement
def formatting_func(samples):
    output_texts = []
    for i in range(len(samples['instruction'])):
        # Added a space before the EOS and ensured proper spacing
        text = (f"### Instruction:\n{samples['instruction'][i]}\n\n"
                f"### Input:\n{samples['input'][i]}\n\n"
                f"### Response:\n{samples['output'][i]} {tokenizer.eos_token}")
        output_texts.append(text)
    return tokenizer(output_texts, truncation=True, max_length=512, padding="max_length")

data = dataset['train'].map(formatting_func, batched=True)

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_patient_info.json:   0%|          | 0.00/3.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

Map:   0%|          | 0/5942 [00:00<?, ? examples/s]

## Model Training and Testing

Now we train the model usig the transformers library. Before doing so, we set the tokenizer to be the end of sequence tokens since it is required by our model. Your goal here is to tune the paramaters until you get a running model on a single colab GPU.

In [ ]:
# Setting the tokenizer padding to be 'eos' tokens
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

trainer = transformers.Trainer(
    model=model,
    train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        warmup_steps=50,
        max_steps=100,
        learning_rate=3e-5,
        bf16=True,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        weight_decay=0.01
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# This silences the warnings
model.config.use_cache = False

# Train the model!
trainer.train()
#Test Your Zaka



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,20.056622
2,25.062618
3,22.184044
4,21.656315
5,23.883781
6,23.220203
7,21.673651
8,21.253166
9,22.901772
10,22.932308


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:323: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=100, training_loss=7.357262880802154, metrics={'train_runtime': 559.5645, 'train_samples_per_second': 1.43, 'train_steps_per_second': 0.179, 'total_flos': 4.98361228591104e+16, 'train_loss': 7.357262880802154, 'epoch': 0.13458950201884254})

Explain 4 of the training arguments you used in your Trainer, how they are used, and what do they represent

* learning_rate=3e-5 : This is the speed of training. Because we are working with a massive 20B model, we use a small number so that it will learn the medical data.

* weight_decay=0.01 : This is a penalty for making weights too large. It will force the model to stay simple and prevent it from just memorizing the medical data word for word.

* optim="paged_adamw_8bit" : This is the optimizer and it calculate the weight changes. "paged" is safety feature, if the GPU run out of memory, it will make computations temporarily on RAM.

* output_dir="outputs" : This is where the model will be saved after training is finished.

**Test your Zaka**

We now save our model as a pretrained version so that we can set the LoRA configurations. This model will be saved to a separate folder on the next block.

In [ ]:
#Test Your Zaka

saved_model = model
# .modules if hasattr(model, "save_pretrained") else model.module
saved_model.save_pretrained("outputs")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:323: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Before testing our model, we have to get the LoRA configs from our pre-trained model and set them to our new model using the get_peft_model() function.

In [ ]:
#Test Your Zaka

lora_configs = LoraConfig.from_pretrained("outputs")
model = get_peft_model(model, lora_configs)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'EleutherAI/gpt-neox-20b' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:301: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


We need to set our prompt as a variable, and also our device currently in use.

In [ ]:
#Test Your Zaka

prompt = "### Instruction:\nAnswer the medical question truthfully.\n\n### Input:\nWhat are the main symptoms of food poisoning?\n\n### Response:\n"
device = "cuda"

Finally, we will make our LLM generate text based on the data. First we user the tokenizer() function on our prompt.

In [ ]:
#Test Your Zaka

inputs = tokenizer(prompt, return_tensors="pt").to(device)

Let's now use the generate() function on our model, and print the decoded version of our output.

In [ ]:
outputs = model.generate(**inputs,
                         max_new_tokens=50,
                         do_sample=True,
                         temperature=0.7,
                         repetition_penalty=1.1,
                         top_p=0.9,
                         pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=tokenizer.eos_token_id
                         )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Answer the medical question truthfully.

### Input:
What are the main symptoms of food poisoning?

### Response:
Diamond Creek Crewing their own caption 1 = BytePtrFromString(1}\left\{ {\boldmath{$X$, $D3*g\times{\oddsidemargin}{-69pt]{minimal} else if you want to be found that he had a large enough time
